# Notebook 01 — Data Quality & Data Preparation

## Brazilian E-Commerce Public Dataset by Olist

**Objetivo:** explorar as 9 bases disponíveis, avaliar qualidade e consistência, documentar os relacionamentos e preparar uma camada analítica confiável para as análises de **pagamentos, RFM, recompra, retenção/coortes e propensão à recompra**.

### Perguntas deste notebook

1. Qual é a estrutura e o período dos dados?
2. Existem nulos, duplicidades ou problemas de chave?
3. Como as tabelas se relacionam?
4. Quais cuidados são necessários para evitar dupla contagem de receita?
5. Qual será a definição de uma compra válida para as análises de comportamento?
6. Quais tabelas analíticas serão produzidas para os próximos notebooks?

> **Princípio de governança:** este notebook não altera os arquivos brutos. Toda transformação é feita em memória e, quando necessário, salva uma camada `processed/`.


## 1. Configuração do projeto

A estrutura esperada no GitHub é:

```text
olist-customer-analytics/
├── data/
│   ├── raw/
│   └── processed/
├── notebooks/
├── src/
├── reports/
└── presentation/
```

Coloque os CSVs originais da Olist em `data/raw/`.

Se estiver executando este notebook dentro da pasta `notebooks/`, os caminhos relativos abaixo funcionarão automaticamente.


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

# Caminhos relativos ao notebook
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT.resolve())
print("RAW_DIR:", RAW_DIR.resolve())
print("RAW_DIR existe?", RAW_DIR.exists())


In [ ]:
# Arquivos esperados
expected_files = {
    "customers": "olist_customers_dataset.csv",
    "geolocation": "olist_geolocation_dataset.csv",
    "order_items": "olist_order_items_dataset.csv",
    "payments": "olist_order_payments_dataset.csv",
    "reviews": "olist_order_reviews_dataset.csv",
    "orders": "olist_orders_dataset.csv",
    "products": "olist_products_dataset.csv",
    "sellers": "olist_sellers_dataset.csv",
    "translation": "product_category_name_translation.csv",
}

missing = [f for f in expected_files.values() if not (RAW_DIR / f).exists()]

if missing:
    raise FileNotFoundError(
        "Arquivos ausentes em data/raw/:\n- " + "\n- ".join(missing)
    )

print(f"{len(expected_files)} arquivos encontrados.")


## 2. Carregamento das nove bases

In [ ]:
# Leitura das bases
customers = pd.read_csv(RAW_DIR / expected_files["customers"])
geolocation = pd.read_csv(RAW_DIR / expected_files["geolocation"])
order_items = pd.read_csv(RAW_DIR / expected_files["order_items"])
payments = pd.read_csv(RAW_DIR / expected_files["payments"])
reviews = pd.read_csv(RAW_DIR / expected_files["reviews"])
orders = pd.read_csv(RAW_DIR / expected_files["orders"])
products = pd.read_csv(RAW_DIR / expected_files["products"])
sellers = pd.read_csv(RAW_DIR / expected_files["sellers"])
translation = pd.read_csv(RAW_DIR / expected_files["translation"])

tables = {
    "customers": customers,
    "geolocation": geolocation,
    "order_items": order_items,
    "payments": payments,
    "reviews": reviews,
    "orders": orders,
    "products": products,
    "sellers": sellers,
    "translation": translation,
}

summary = pd.DataFrame({
    "table": list(tables.keys()),
    "rows": [df.shape[0] for df in tables.values()],
    "columns": [df.shape[1] for df in tables.values()],
})

summary


### 2.1 Dicionário rápido das tabelas

In [ ]:
table_description = pd.DataFrame({
    "table": [
        "orders", "customers", "order_items", "payments", "reviews",
        "products", "sellers", "geolocation", "translation"
    ],
    "granularity": [
        "1 linha por pedido",
        "1 linha por customer_id",
        "1 linha por item de pedido",
        "1 linha por pagamento/sequência do pedido",
        "1 linha por avaliação",
        "1 linha por produto",
        "1 linha por vendedor",
        "1 linha por registro de CEP",
        "1 linha por categoria"
    ],
    "business_role": [
        "Status e ciclo do pedido",
        "Identificação do cliente e customer_unique_id",
        "Produtos, vendedores, preços e frete",
        "Meio de pagamento, parcelas e valor",
        "Satisfação do cliente",
        "Dimensões de produto",
        "Dimensão de vendedor",
        "Apoio geográfico",
        "Tradução de categorias"
    ]
})

table_description


## 3. Tipos de dados e datas

In [ ]:
# Conversão das colunas temporais
date_cols_orders = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
]

for col in date_cols_orders:
    orders[col] = pd.to_datetime(orders[col], errors="coerce")

for col in ["shipping_limit_date"]:
    order_items[col] = pd.to_datetime(order_items[col], errors="coerce")

for col in ["review_creation_date", "review_answer_timestamp"]:
    reviews[col] = pd.to_datetime(reviews[col], errors="coerce")

orders.dtypes.to_frame("dtype")


In [ ]:
date_summary = pd.DataFrame({
    "column": date_cols_orders,
    "min": [orders[c].min() for c in date_cols_orders],
    "max": [orders[c].max() for c in date_cols_orders],
    "nulls": [orders[c].isna().sum() for c in date_cols_orders],
})

date_summary


### Observação metodológica

Para as análises de comportamento do cliente, a data-base principal será **`order_purchase_timestamp`**, pois representa o momento da compra.

Datas posteriores, como aprovação e entrega, serão utilizadas em análises complementares e não devem ser usadas para criar features que não estariam disponíveis no momento da compra.


## 4. Qualidade: nulos

In [ ]:
null_summary = pd.DataFrame({
    "table": list(tables.keys()),
    "rows": [len(df) for df in tables.values()],
    "null_cells": [int(df.isna().sum().sum()) for df in tables.values()],
    "null_pct_cells": [
        float(df.isna().mean().mean() * 100) for df in tables.values()
    ],
})

null_summary.sort_values("null_cells", ascending=False)


In [ ]:
# Detalhamento dos campos com nulos
null_details = []

for name, df in tables.items():
    for col in df.columns:
        n = int(df[col].isna().sum())
        if n > 0:
            null_details.append({
                "table": name,
                "column": col,
                "nulls": n,
                "null_pct": n / len(df) * 100
            })

null_details = pd.DataFrame(null_details).sort_values(
    ["table", "nulls"], ascending=[True, False]
)

null_details


### Interpretação inicial

Os nulos não são necessariamente erros.

Exemplos:

- `order_approved_at`, datas de entrega: podem ser nulos porque o pedido não avançou no fluxo.
- comentários de avaliação: ausência de comentário não significa ausência de avaliação.
- atributos físicos de produto: podem estar ausentes para alguns produtos.

A decisão de tratamento deve ser feita **por campo e por regra de negócio**, e não simplesmente substituir todos os nulos por zero.


## 5. Duplicidades

In [ ]:
duplicate_summary = pd.DataFrame({
    "table": list(tables.keys()),
    "duplicate_rows": [int(df.duplicated().sum()) for df in tables.values()],
})

duplicate_summary


In [ ]:
# Duplicidades nas chaves esperadas
key_checks = {
    "orders.order_id": (orders, ["order_id"]),
    "customers.customer_id": (customers, ["customer_id"]),
    "customers.customer_unique_id": (customers, ["customer_unique_id"]),
    "products.product_id": (products, ["product_id"]),
    "sellers.seller_id": (sellers, ["seller_id"]),
    "order_items.order_id + order_item_id": (
        order_items, ["order_id", "order_item_id"]
    ),
    "payments.order_id + payment_sequential": (
        payments, ["order_id", "payment_sequential"]
    ),
    "reviews.review_id": (reviews, ["review_id"]),
    "translation.product_category_name": (
        translation, ["product_category_name"]
    ),
}

key_results = []

for key_name, (df, cols) in key_checks.items():
    key_results.append({
        "key": key_name,
        "duplicates": int(df.duplicated(subset=cols).sum()),
        "unique_combinations": int(df[cols].drop_duplicates().shape[0])
    })

pd.DataFrame(key_results)


### Ponto de atenção: avaliações

A chave `review_id` não é necessariamente única no dataset. Portanto, **não devemos assumir 1 review_id = 1 pedido**.

Para análises de satisfação agregadas ao nível do pedido/cliente, será necessário definir uma regra de deduplicação ou agregação. Essa decisão será documentada no notebook de reviews, caso essa dimensão seja utilizada.


## 6. Integridade referencial

In [ ]:
def anti_join_count(left, left_key, right, right_key):
    left_values = set(left[left_key].dropna().unique())
    right_values = set(right[right_key].dropna().unique())
    return len(left_values - right_values)

integrity = [
    {
        "relationship": "orders.customer_id -> customers.customer_id",
        "orphans": anti_join_count(orders, "customer_id", customers, "customer_id")
    },
    {
        "relationship": "order_items.order_id -> orders.order_id",
        "orphans": anti_join_count(order_items, "order_id", orders, "order_id")
    },
    {
        "relationship": "order_items.product_id -> products.product_id",
        "orphans": anti_join_count(order_items, "product_id", products, "product_id")
    },
    {
        "relationship": "order_items.seller_id -> sellers.seller_id",
        "orphans": anti_join_count(order_items, "seller_id", sellers, "seller_id")
    },
    {
        "relationship": "payments.order_id -> orders.order_id",
        "orphans": anti_join_count(payments, "order_id", orders, "order_id")
    },
    {
        "relationship": "reviews.order_id -> orders.order_id",
        "orphans": anti_join_count(reviews, "order_id", orders, "order_id")
    },
]

pd.DataFrame(integrity)


## 7. Perfil dos pedidos

Antes de definir o universo de clientes, precisamos entender os status existentes.

Para as análises de **comportamento de compra e RFM**, a regra recomendada é trabalhar inicialmente com pedidos **`delivered`**, porque representam compras concluídas.

Pedidos `canceled`, `unavailable`, `created`, `approved`, `processing`, `invoiced` e `shipped` podem ser analisados separadamente, mas não devem ser tratados automaticamente como compras concluídas.


In [ ]:
order_status = (
    orders["order_status"]
    .value_counts()
    .rename_axis("order_status")
    .reset_index(name="orders")
)

order_status["share"] = order_status["orders"] / order_status["orders"].sum()

order_status


In [ ]:
# Período completo
period = pd.Series({
    "first_purchase": orders["order_purchase_timestamp"].min(),
    "last_purchase": orders["order_purchase_timestamp"].max(),
    "days": (
        orders["order_purchase_timestamp"].max()
        - orders["order_purchase_timestamp"].min()
    ).days
})

period.to_frame("value")


## 8. Perfil de clientes e `customer_unique_id`

In [ ]:
customer_profile = pd.DataFrame({
    "customer_id": [customers["customer_id"].nunique()],
    "customer_unique_id": [customers["customer_unique_id"].nunique()],
    "customer_id_rows": [len(customers)],
})

customer_profile


In [ ]:
# Quantos customer_id pertencem a cada customer_unique_id?
unique_id_frequency = (
    customers.groupby("customer_unique_id")["customer_id"]
    .nunique()
    .value_counts()
    .sort_index()
    .rename_axis("customer_ids_per_unique_customer")
    .reset_index(name="unique_customers")
)

unique_id_frequency


### Regra fundamental para este projeto

Para RFM, recompra e coortes, o identificador analítico deve ser:

**`customer_unique_id`**

e não `customer_id`.

`customer_id` representa a identificação do registro do cliente em um pedido, enquanto `customer_unique_id` permite consolidar o histórico de compras do mesmo consumidor ao longo de diferentes pedidos.


## 9. Pagamentos: cuidado com dupla contagem

In [ ]:
payment_order_check = (
    payments.groupby("order_id")
    .agg(
        payment_rows=("payment_sequential", "count"),
        payment_total=("payment_value", "sum"),
    )
)

payment_order_check.describe()


In [ ]:
payment_type_summary = (
    payments.groupby("payment_type")
    .agg(
        payment_rows=("payment_type", "size"),
        orders=("order_id", "nunique"),
        payment_value=("payment_value", "sum"),
        avg_payment=("payment_value", "mean"),
    )
    .sort_values("payment_value", ascending=False)
)

payment_type_summary["share_value"] = (
    payment_type_summary["payment_value"]
    / payment_type_summary["payment_value"].sum()
)

payment_type_summary


### Regra de governança financeira

`payments` possui granularidade de **pagamento**, não de pedido.

Portanto:

- para saber o valor total pago por pedido: `groupby("order_id")["payment_value"].sum()`;
- para saber participação por meio de pagamento: é válido trabalhar na granularidade dos registros de pagamento;
- para métricas de pedido/cliente, primeiro agregamos pagamentos para `order_id` e só depois fazemos o join com `orders`.

Isso evita inflar receita ao juntar diretamente `orders`, `order_items` e `payments`.


## 10. Itens de pedido e risco de multiplicação de linhas

In [ ]:
items_per_order = (
    order_items.groupby("order_id")
    .agg(
        items=("order_item_id", "count"),
        product_value=("price", "sum"),
        freight_value=("freight_value", "sum"),
    )
)

items_per_order.describe()


In [ ]:
# Demonstração da granularidade
granularity_check = pd.DataFrame({
    "dataset": ["orders", "order_items", "payments", "reviews"],
    "rows": [len(orders), len(order_items), len(payments), len(reviews)],
    "distinct_orders": [
        orders["order_id"].nunique(),
        order_items["order_id"].nunique(),
        payments["order_id"].nunique(),
        reviews["order_id"].nunique(),
    ],
})

granularity_check


## 11. Definição do universo analítico

Para o projeto de **Comportamento e Pagamentos**, vamos criar duas camadas:

### Camada A — `orders_base`

Uma linha por pedido, contendo:

- `order_id`
- `customer_unique_id`
- `order_purchase_timestamp`
- `order_status`
- valor total dos pagamentos
- valor total dos produtos
- frete
- quantidade de itens

### Camada B — `customer_orders`

Uma linha por pedido + cliente, já consolidada pelo `customer_unique_id`.

Essa segunda camada será a principal entrada dos notebooks de RFM, recompra e coortes.


In [ ]:
# 1) Mapeia customer_id -> customer_unique_id
customer_map = customers[
    ["customer_id", "customer_unique_id"]
].drop_duplicates()

# 2) Agrega pagamentos ao nível do pedido
payments_order = (
    payments.groupby("order_id", as_index=False)
    .agg(
        payment_value=("payment_value", "sum"),
        payment_count=("payment_sequential", "count"),
    )
)

# 3) Agrega itens ao nível do pedido
items_order = (
    order_items.groupby("order_id", as_index=False)
    .agg(
        item_count=("order_item_id", "count"),
        product_value=("price", "sum"),
        freight_value=("freight_value", "sum"),
        distinct_products=("product_id", "nunique"),
        distinct_sellers=("seller_id", "nunique"),
    )
)

# 4) Junta as dimensões ao nível do pedido
orders_base = (
    orders
    .merge(customer_map, on="customer_id", how="left", validate="one_to_one")
    .merge(payments_order, on="order_id", how="left", validate="one_to_one")
    .merge(items_order, on="order_id", how="left", validate="one_to_one")
)

orders_base.shape, orders_base["order_id"].nunique()


In [ ]:
# Conferência da camada orders_base
orders_base_check = {
    "rows": len(orders_base),
    "distinct_orders": orders_base["order_id"].nunique(),
    "null_customer_unique_id": orders_base["customer_unique_id"].isna().sum(),
    "null_payment_value": orders_base["payment_value"].isna().sum(),
    "null_product_value": orders_base["product_value"].isna().sum(),
}

pd.Series(orders_base_check)


### Tratamento dos nulos na camada consolidada

Pedidos sem pagamento ou sem item não devem ser silenciosamente convertidos em compras de valor zero.

Para métricas financeiras, manteremos os nulos inicialmente e investigaremos esses casos. Quando uma métrica exigir um valor numérico completo, a regra de tratamento será explícita.


## 12. Universo recomendado para RFM

In [ ]:
# Universo de compras concluídas
customer_orders = (
    orders_base[
        orders_base["order_status"].eq("delivered")
        & orders_base["customer_unique_id"].notna()
        & orders_base["order_purchase_timestamp"].notna()
    ]
    .copy()
)

customer_orders["order_purchase_date"] = (
    customer_orders["order_purchase_timestamp"].dt.normalize()
)

print("Pedidos entregues:", customer_orders["order_id"].nunique())
print("Clientes únicos:", customer_orders["customer_unique_id"].nunique())
print("Período:", customer_orders["order_purchase_timestamp"].min(), "até",
      customer_orders["order_purchase_timestamp"].max())


In [ ]:
# Métricas básicas por cliente
customer_summary = (
    customer_orders
    .groupby("customer_unique_id", as_index=False)
    .agg(
        first_purchase=("order_purchase_timestamp", "min"),
        last_purchase=("order_purchase_timestamp", "max"),
        frequency=("order_id", "nunique"),
        monetary=("payment_value", "sum"),
        avg_order_value=("payment_value", "mean"),
        total_items=("item_count", "sum"),
    )
)

customer_summary.head()


## 13. Primeiras métricas de recompra

Ainda não é o notebook final de recompra; aqui fazemos apenas uma validação estrutural da base.



In [ ]:
repurchase_summary = (
    customer_summary["frequency"]
    .value_counts()
    .sort_index()
    .rename_axis("orders_per_customer")
    .reset_index(name="customers")
)

repurchase_summary["customer_share"] = (
    repurchase_summary["customers"] / repurchase_summary["customers"].sum()
)

repurchase_summary.head(15)


In [ ]:
repurchase_rate = (
    customer_summary["frequency"].gt(1).mean()
)

single_purchase_share = (
    customer_summary["frequency"].eq(1).mean()
)

pd.Series({
    "customers": len(customer_summary),
    "single_purchase_share": single_purchase_share,
    "repurchase_rate": repurchase_rate,
    "avg_frequency": customer_summary["frequency"].mean(),
    "median_frequency": customer_summary["frequency"].median(),
})


## 14. Exportação da camada processada

Salvamos uma tabela consolidada por pedido para que os próximos notebooks não precisem repetir todos os joins.

Também salvamos a tabela de cliente para RFM/recompra/coortes.

> Em um repositório público, **não suba os CSVs brutos**. O ideal é manter os dados fora do Git e documentar como obtê-los.


In [ ]:
# Exporta camadas analíticas
orders_base_path = PROCESSED_DIR / "orders_base.csv"
customer_summary_path = PROCESSED_DIR / "customer_summary_base.csv"

orders_base.to_csv(orders_base_path, index=False)
customer_summary.to_csv(customer_summary_path, index=False)

print("Salvo:")
print(orders_base_path)
print(customer_summary_path)


## 15. Checklist de qualidade

Ao final deste notebook, devemos conseguir marcar:

- [x] As 9 tabelas foram carregadas.
- [x] Estrutura e granularidade foram documentadas.
- [x] Tipos de data foram tratados.
- [x] Nulos foram identificados.
- [x] Duplicidades foram investigadas.
- [x] Integridade referencial foi verificada.
- [x] `customer_unique_id` foi definido como chave analítica de cliente.
- [x] Pagamentos foram agregados antes do join com pedidos.
- [x] Itens foram agregados antes do join com pedidos.
- [x] Pedidos entregues foram separados para análises de comportamento.
- [x] Camadas `orders_base` e `customer_summary_base` foram preparadas.

### Próximo notebook

**Notebook 02 — Payment Analysis**

Objetivo: responder como os clientes pagam, como o parcelamento se relaciona com o valor das compras e quais comportamentos de pagamento são relevantes para a visão executiva.
